# Baysor outputs → AnnData (.h5ad) — Stroke MERSCOPE (Fan / CG)

Collects the per-region Baysor outputs written by `baysor_stroke_batch.ipynb` and turns each into an
`.h5ad`:

- builds the **cell × gene** count matrix from `segmentation.csv` (transcripts with a `cell` assignment),
- attaches per-cell metadata from `segmentation_cell_stats.csv` (centroids, area, n_transcripts, …) to `.obs`,
- stores the centroid in `.obsm['spatial']`,
- writes one `<sample>.h5ad` per region and an optional concatenated `stroke_all.h5ad`.

**Kernel:** `sc` (the conda env with scanpy / pandas / numpy).

In [2]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad

sc.settings.verbosity = 1

# Must match `output_root` + `param_tag` from baysor_stroke_batch.ipynb
seg_root  = Path("/Volumes/T7/Stroke_merscop_Fan_CG/baysor_segmentation")
param_tag = "m50_s4"

# Where the .h5ad files are written
out_dir = Path("/Volumes/T7/Stroke_merscop_Fan_CG/h5ad")
out_dir.mkdir(parents=True, exist_ok=True)

assert seg_root.exists(), f"Segmentation root not found: {seg_root}"

## 1. Find finished regions

In [3]:
def find_outputs(root: Path, tag: str):
    found = []
    for seg in sorted(root.glob(f"*/{tag}/segmentation_segmentation.csv")):
        if seg.name.startswith("._"):
            continue
        sample = seg.parent.parent.name           # <root>/<sample>/<tag>/segmentation.csv
        stats = seg.parent / "segmentation_cell_stats.csv"
        if seg.stat().st_size > 0 and stats.exists():
            found.append({"sample": sample, "seg": seg, "stats": stats})
    return found

outputs = find_outputs(seg_root, param_tag)
print(f"Found {len(outputs)} finished regions:")
for o in outputs:
    print("  ", o["sample"])

Found 33 finished regions:
   1005_glast_dmcao_5d_1
   1005_glast_dmcao_5d_2
   1149_glast_dmcao_clp_7d_1
   1149_glast_dmcao_clp_7d_2
   1149_glast_dmcao_clp_7d_3
   1149_glast_dmcao_clp_7d_4
   1149_glast_dmcao_clp_7d_5
   1151_glast_dmcao_lip_7d_1
   1151_glast_dmcao_lip_7d_2
   122_mrc1_dcmao_1
   122_mrc1_dcmao_2
   122_mrc1_dcmao_3
   139_mrc1_dmcao_5d_1
   139_mrc1_dmcao_5d_2
   139_mrc1_dmcao_5d_3
   141_mrc1_dmcao_14d_1
   141_mrc1_dmcao_14d_2
   141_mrc1_dmcao_14d_3
   141_mrc1_dmcao_14d_4
   182_mrc1_uninjured
   26_mrc1_tcmao_14d
   27_mrc1_uninjured
   465_col1a1_tmcao_5d
   467_col1a1_uninjured
   51_mrc1_dcmao_14d
   537_col1a1_dmcao_5d
   54_mrc1_dcmao_5d
   58_mrc1_tcmao_14d
   609_glast_dmcao_14d
   60_mrc1_dcmao_5d
   619_col1a1_tmcao_14d
   700_glast_dmcao_5d
   751_col1a1_tmcao_5d


## 2. Build one AnnData per region

Baysor's `segmentation.csv` is one row per transcript with a `cell` column (0 / empty = background noise,
which we drop). We cross-tabulate transcripts into a cell × gene matrix and align it with the cell-stats table.

In [12]:
def build_adata(sample: str, seg: Path, stats: Path) -> ad.AnnData:
    tx = pd.read_csv(seg)
    tx = tx[tx["cell"].notna()]
    tx["cell"] = tx["cell"].astype(str)
    # Drop the background label (Baysor uses 0 / empty for noise transcripts)
    tx = tx[~tx["cell"].isin(["0", "", "nan"])]
    tx["gene"] = tx["gene"].astype(str)

    counts = pd.crosstab(tx["cell"], tx["gene"])

    st = pd.read_csv(stats)
    st["cell"] = st["cell"].astype(str)
    st = st.set_index("cell").reindex(counts.index)

    adata = ad.AnnData(
        X=counts.values.astype(np.float32),
        obs=st,
        var=pd.DataFrame(index=counts.columns),
    )
    adata.obs_names = [f"{sample}_{c}" for c in counts.index]
    adata.obs["sample"] = sample

    # Spatial centroid -> .obsm['spatial'] (Baysor stats name the centroid x/y)
    xcol = next((c for c in ("x", "centroid_x", "global_x") if c in adata.obs), None)
    ycol = next((c for c in ("y", "centroid_y", "global_y") if c in adata.obs), None)
    if xcol and ycol:
        adata.obsm["spatial"] = adata.obs[[xcol, ycol]].to_numpy(dtype=float)
    return adata


adatas = {}
for o in outputs:
    a = build_adata(o["sample"], o["seg"], o["stats"])
    adatas[o["sample"]] = a
    out_path = out_dir / f"{o['sample']}.h5ad"
    a.write_h5ad(out_path)
    print(f"{o['sample']:40s}  cells={a.n_obs:6d}  genes={a.n_vars:4d}  ->  {out_path.name}")

1005_glast_dmcao_5d_1                     cells= 60283  genes= 500  ->  1005_glast_dmcao_5d_1.h5ad
1005_glast_dmcao_5d_2                     cells= 80060  genes= 500  ->  1005_glast_dmcao_5d_2.h5ad
1149_glast_dmcao_clp_7d_1                 cells=136505  genes= 500  ->  1149_glast_dmcao_clp_7d_1.h5ad
1149_glast_dmcao_clp_7d_2                 cells=151686  genes= 500  ->  1149_glast_dmcao_clp_7d_2.h5ad
1149_glast_dmcao_clp_7d_3                 cells=124845  genes= 500  ->  1149_glast_dmcao_clp_7d_3.h5ad
1149_glast_dmcao_clp_7d_4                 cells= 46302  genes= 500  ->  1149_glast_dmcao_clp_7d_4.h5ad
1149_glast_dmcao_clp_7d_5                 cells= 89511  genes= 500  ->  1149_glast_dmcao_clp_7d_5.h5ad
1151_glast_dmcao_lip_7d_1                 cells= 83351  genes= 500  ->  1151_glast_dmcao_lip_7d_1.h5ad
1151_glast_dmcao_lip_7d_2                 cells= 46312  genes= 500  ->  1151_glast_dmcao_lip_7d_2.h5ad
122_mrc1_dcmao_1                          cells= 66000  genes= 500  ->  122_mrc1_

## 3. (Optional) concatenate all regions into one object

Outer join on genes so panels with small differences still align; `sample` in `.obs` keeps regions separable.

In [13]:
if adatas:
    combined = ad.concat(adatas, join="outer", label="sample", index_unique=None)
    combined.X = np.nan_to_num(combined.X)
    combined_path = out_dir / "stroke_all.h5ad"
    combined.write_h5ad(combined_path)
    print(combined)
    print("\nWrote", combined_path)
else:
    print("No finished regions yet — run baysor_stroke_batch.ipynb first.")

AnnData object with n_obs × n_vars = 2254032 × 500
    obs: 'x', 'y', 'z', 'cluster', 'n_transcripts', 'density', 'elongation', 'area', 'avg_confidence', 'avg_assignment_confidence', 'max_cluster_frac', 'lifespan', 'sample'
    obsm: 'spatial'

Wrote /Volumes/T7/Stroke_merscop_Fan_CG/h5ad/stroke_all.h5ad
